# Part E: Audio-Specific Architectures

---

## Why Architecture Matters

You now have all the building blocks: Conv1D, dilated convolutions, strided convolutions, transposed convolutions, LMF features. Architecture is about how you *assemble* these blocks into a system that solves a specific problem.

Three architectures dominate audio deep learning. Each was designed to solve a specific weakness of the one before it.

---

## Architecture 1: U-Net (Encoder-Decoder with Skip Connections)

### The Problem It Solves

Imagine you want to separate two speakers from a mixed audio file. Your network needs to:
1. **Compress** the audio into a compact representation (understand what's happening)
2. **Reconstruct** the separated audio at full resolution (produce the output)

A simple encoder-decoder does this:

```
Input (48000,) → Encoder → Bottleneck (small) → Decoder → Output (48000,)
```

But here's the problem: the bottleneck is small by design — it forces the network to compress. In doing so, it loses fine-grained details (exact sample values, precise timing). The decoder has to reconstruct these details from a compressed code, which is hard.

### The Skip Connection Solution

U-Net's insight: the encoder saw the full-resolution signal. Why not give that information directly to the decoder instead of forcing it to reconstruct it?

Skip connections pass the encoder's intermediate representations directly to the corresponding decoder layer:

```
Input
  ↓
Encoder Layer 1 ──────────────────────────→ (+) Decoder Layer 4
  ↓                                              ↑
Encoder Layer 2 ──────────────────────→ (+) Decoder Layer 3
  ↓                                              ↑
Encoder Layer 3 ──────────────────→ (+) Decoder Layer 2
  ↓                                              ↑
Encoder Layer 4 → Bottleneck → Decoder Layer 1
```

Each decoder layer receives two inputs: the upsampled bottleneck representation AND the skip connection from the corresponding encoder layer. The `(+)` is typically concatenation along the channel dimension.

**Why this works:** The encoder layers have different "views" of the signal — early layers see fine-grained local patterns, late layers see coarse global structure. By giving each decoder layer access to the corresponding encoder view, the decoder can reconstruct detail at every scale.

### Where You'll See This

Conv-TasNet's overall shape is U-Net style:
```
Raw waveform → Conv1d encoder → TCN stack → masks → ConvTranspose1d decoder → separated audio
```

The masks produced by the TCN are multiplied with the encoder output, then decoded. The encoder-decoder structure with the TCN in the middle is the U-Net pattern applied to audio.

Full speech enhancement models (Demucs, SEGAN) use explicit skip connections between encoder and decoder layers.

---

## Architecture 2: Temporal Convolutional Network (TCN)

You've already built a mini version of this in Part D. Now let's understand it as a complete architecture.

### What a TCN Is

A TCN is a stack of dilated, causal Conv1D layers designed specifically to model sequences. The key design decisions and why they exist:

**1. Dilated convolutions with exponentially increasing dilation**

You know this from Part C1 — gives large receptive field cheaply.

```
Dilation: 1, 2, 4, 8, 16, 32, 64, 128
```

**2. Residual connections**

Each TCN block adds its input to its output:

```
output = F(x) + x
```

Where `F(x)` is the dilated conv operation. This is borrowed from ResNets and solves a critical problem: with 24 stacked layers, gradients vanish before reaching the early layers during backpropagation. The residual connection gives gradients a "shortcut" path back through the network.

Without residual connections, deep TCNs don't train. With them, you can stack as many layers as you want.

**3. The full TCN block structure**

Each block in Conv-TasNet looks like:

```
Input (x)
  ↓
Conv1d 1×1  (channel mixing, expand)
  ↓
PReLU
  ↓
Normalization
  ↓
Depthwise Conv1d (temporal filtering, dilated, causal)
  ↓
PReLU
  ↓
Normalization
  ↓
Conv1d 1×1  (channel mixing, compress)
  ↓
(+) ← add x (residual connection)
  ↓
Output
```

The 1×1 convolutions at the start and end are the "pointwise" part of depthwise-separable convolution — they mix channels without touching the time dimension. The depthwise conv in the middle does the temporal filtering. You built all of these pieces separately in Part C.

**4. Repeated stacks**

One stack of 8 dilated layers gives ~32ms receptive field (after the encoder). Three stacks of 8 layers gives ~1.5 seconds. Conv-TasNet uses 3 repetitions:

```
Stack 1: dilation 1,2,4,8,16,32,64,128
Stack 2: dilation 1,2,4,8,16,32,64,128  (same structure, different weights)
Stack 3: dilation 1,2,4,8,16,32,64,128
```

Each stack learns to look at the same timescales but with different learned patterns, building increasingly abstract representations.

---

## Architecture 3: Self-Attention for Long-Range Dependencies

### The Fundamental Limit of Conv

Even with a 1.5-second receptive field, TCN has a hard ceiling — any two positions more than 1.5 seconds apart have zero influence on each other. For a 30-second recording, the network at position 0 is completely blind to what happens at position 25 seconds.

For **speaker diarization** this is a serious problem. To assign a consistent speaker label throughout 30 seconds, the network must be able to compare voice characteristics at any two positions regardless of distance. The speaker at second 28 might be the same person as second 2 — a TCN can't know that.

### What Attention Does

Self-attention lets every position in the sequence directly attend to every other position. No locality constraint, no fixed receptive field.

For each position `t`, attention computes:
- **Query (Q):** "what am I looking for?"
- **Key (K):** "what do I contain?"
- **Value (V):** "what information do I provide?"

The output at position `t` is a weighted sum of all Values, where the weight between position `t` and position `s` is determined by how well Query(t) matches Key(s):

```
Attention(t, s) = softmax( Q(t) · K(s) / √d )
Output(t) = Σ_s  Attention(t, s) × V(s)
```

where `d` is dimension of your features

In plain English: position `t` looks at every other position, decides how relevant each one is, and combines their information proportionally.

**The result:** position 2 seconds can directly influence position 28 seconds in a single layer. No stacking required for long-range dependencies.

### The Cost

Attention is expensive. For a sequence of length `T`, attention requires computing `T × T` pairwise scores. For a 30-second clip at 10ms frames, `T = 3000`:

```
TCN:       O(T)   computation per layer — linear in sequence length
Attention: O(T²)  computation — quadratic in sequence length
```

3000² = 9,000,000 operations per attention layer. This is why EEND applies attention to **LMF features** (T=3000) rather than raw waveforms (T=480,000) — applying attention directly to raw audio would be computationally infeasible.

### How EEND Combines Both

EEND's architecture is:

```
LMF features (80, 3000)
  ↓
Linear projection → (256, 3000)   ← map to model dimension
  ↓
Positional encoding                ← tell attention where each frame is
  ↓
Transformer Encoder × 4 layers:
  - Multi-head self-attention      ← global context
  - Feed-forward network           ← local processing
  - Layer normalization
  ↓
Linear → (num_speakers, 3000)      ← per-frame speaker activity
  ↓
Sigmoid                            ← probability each speaker is active
```

The Transformer gives EEND its global context — it can compare any frame to any other frame directly.

---

## When to Use Which

```
                U-Net          TCN              Attention
─────────────────────────────────────────────────────────
Context         Local          Local/Medium     Global
Sequence len    Any            Any              Short-medium (cost is T²)
Typical use     Separation     Separation       Diarization, recognition
Receptive field Encoder depth  Stack × dilation Full sequence
Causal?         Optional       Yes (padding)    Optional (masking)
Your system     Conv-TasNet    Conv-TasNet      EEND
─────────────────────────────────────────────────────────
```

**The design principle behind the split:**

Conv-TasNet separates speech — a local task. To separate two overlapping voices, you mainly need to look at a 1-2 second window. TCN with large receptive field is sufficient.

EEND diarizes speech — a global task. To label who speaks when across 30 seconds, you need to compare frames from anywhere in the recording. Attention is necessary.

This is why your final system uses both: Conv-TasNet for separation, EEND for diarization, and they operate on different feature representations (raw waveform vs LMF).

---

## TODO: Build a Mini EEND Encoder

**Goal:** Build the front-end of EEND — from LMF features to a sequence of contextualized representations using self-attention. This is the core of what makes EEND work.

**Requirements:**

**Part 1 — Self-attention from scratch:**

Implement a `SelfAttention` module as `nn.Module` with:
- A linear projection for Q, K, V — each maps `(batch, time, d_model)` → `(batch, time, d_model)`
- Scaled dot-product attention: `softmax(QKᵀ / √d_model) × V`
- Output linear projection back to `d_model`

Test it on a `(2, 300, 64)` input (batch=2, time=300, d_model=64). Output shape should be identical to input shape.

**Part 2 — Transformer Encoder Block:**

Build a `TransformerBlock` module containing:
- `SelfAttention`
- A feed-forward network: `Linear(d_model, 4×d_model) → ReLU → Linear(4×d_model, d_model)`
- Layer normalization after each sub-layer
- Residual connection around each sub-layer

The structure is:
```
x = x + SelfAttention(LayerNorm(x))
x = x + FeedForward(LayerNorm(x))
```

**Part 3 — Mini EEND front-end:**

Stack 2 `TransformerBlock`s into a `MiniEEND` module that:
- Takes LMF features `(batch, 80, T)` as input
- Projects to `d_model=64` with a linear layer (you'll need to reshape — attention expects time as the middle dimension, not channels)
- Passes through 2 Transformer blocks
- Projects to `num_speakers=3` outputs
- Applies sigmoid to get per-frame speaker activity probabilities
- Output shape: `(batch, T, 3)` — probability that each speaker is active at each time frame

Test with a real batch from your `LogMelSpeakerDataset`.

**Hints:**
- Attention expects `(batch, time, features)` but your LMF is `(batch, features, time)` — use `.transpose(1, 2)` to swap
- Scaled dot-product: `scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_model)`
- `torch.softmax(scores, dim=-1)` — softmax over the last dimension (which keys are attended to)
- `nn.LayerNorm(d_model)` normalizes over the last dimension — make sure your tensor has `d_model` as the last dim before applying it
- For the output: you don't need to match speaker labels to specific speakers yet (that's Part F — PIT). Just produce 3 probability values per frame

**Key questions:**
- What is the shape of the attention score matrix `QKᵀ`? What does each entry represent?
- Your sequence length T for 30-second clips is ~1875 frames (at hop=160). What is the size of the attention matrix for one sample? Is this feasible on your MacBook?
- Why do we use LayerNorm here instead of BatchNorm (which you used in Part B)?

---

In [215]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchaudio.transforms as T
import math
from pathlib import Path
import os
import sys
import json
import matplotlib.pyplot as plt

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


In [216]:
class SpeakerDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_path, limit=None):
        # Load manifest JSON
        # Store all entries as self.data
        with open(manifest_path, "r") as f:
            full_data = json.load(f)
        
        if limit:
            self.data = full_data[:limit]
        else:
            self.data = full_data
    
    def __len__(self):
        # Return number of samples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get entry at index idx
        # Load audio from disk
        # Extract features
        # Get label (num_speakers - 1)
        # Return (features, label)
        entry = self.data[idx]
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, _ = load_audio(mixture_path)
        mixture_tensor = torch.from_numpy(mixture_audio)
        
        max_len = 48000
        
        if mixture_tensor.size(0) > max_len:
            # Truncate if too long
            mixture_tensor = mixture_tensor[:max_len]
        else:
            # Pad with zeros if too short
            padding = max_len - mixture_tensor.size(0)
            mixture_tensor = torch.nn.functional.pad(mixture_tensor, (0, padding))
        
        
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        return mixture_tensor, label_tensor

In [217]:
train_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "train" / "train_manifest.json"
train_dataset = SpeakerDataset(train_manifest_path)
print(len(train_dataset))

val_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "val" / "val_manifest.json"
val_dataset = SpeakerDataset(val_manifest_path, 2000)
print(len(val_dataset))

10000
2000


In [218]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

### todo 1

part 1

In [219]:
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.linear_Q = nn.Linear(d_model, d_model)
        self.linear_K = nn.Linear(d_model, d_model)
        self.linear_V = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        Q = self.linear_Q(x)
        K = self.linear_K(x)
        V = self.linear_V(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)
        weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(weights, V)
        output = self.out_proj(context)
        return output

In [220]:
sample = torch.randn(2, 300, 64) #(batch, time, d_model)

model_attention = SelfAttention(64)
output = model_attention(sample)
print(output.shape)

assert sample.shape == output.shape, f"shape mismatch! Actual shape: {output.shape}, Expected shape: {sample.shape}"

torch.Size([2, 300, 64])


part 2

In [221]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attention = SelfAttention(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self,x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x

In [222]:
sample = torch.randn(2, 300, 64) #(batch, time, d_model)

model_Transformer = TransformerBlock(64)
output = model_Transformer(sample)
print(output.shape)

assert sample.shape == output.shape, f"shape mismatch! Actual shape: {output.shape}, Expected shape: {sample.shape}"

torch.Size([2, 300, 64])


part 3

In [223]:
class MiniEEND(nn.Module):
    def __init__(self, n_mels, d_model, num_speakers):
        super().__init__()
        self.linearprojection = nn.Linear(n_mels, d_model)
        self.block1 = TransformerBlock(d_model=d_model)
        self.block2 = TransformerBlock(d_model=d_model)
        self.outputlayer = nn.Linear(d_model, num_speakers)
        

    def forward(self,x):
        x = x.transpose(1,2)
        x = self.linearprojection(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.outputlayer(x)
        x = torch.sigmoid(x)
        return x

In [224]:
def torchLMF(audio, n_fft, hop_length):
    torch_lmf = T.MelSpectrogram(
        sample_rate=16000,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=80
        )
    mel_spec = torch_lmf(audio)
    log = torch.log(mel_spec + 1e-8)
    return log

In [225]:
n_fft = 400
hop_length = 160
n_mels = 80
d_model=64
num_speakers = 3

batch, labels = next(iter(train_loader))
features = torchLMF(batch, n_fft=n_fft, hop_length=hop_length)

model_MiniEEND = MiniEEND(n_mels=n_mels, d_model=d_model, num_speakers=num_speakers)

print(f"input shape: {features.shape}")

output = model_MiniEEND(features)

print(f"output shape: {output.shape}")


input shape: torch.Size([32, 80, 301])
output shape: torch.Size([32, 301, 3])


# question: why is the output shape is (batch, T, 3) shouldn't it be (batch, 3) or we are predicting number of speakers per T time frame

Great question — and it reveals something fundamental about how EEND thinks about the problem differently from what you've built so far.

Your current models output `(batch, 3)` — one prediction per clip. That's answering: **"how many speakers are in this recording?"**

EEND outputs `(batch, T, 3)` — one prediction per time frame. That's answering: **"at this specific moment in time, is speaker 1 active? is speaker 2 active? is speaker 3 active?"**

These are completely different tasks.

---

## The Distinction

Your current approach is **clip-level classification:**
```
3-second clip → model → "2 speakers"
```

EEND does **frame-level activity detection:**
```
frame 0   → model → [0.02, 0.95, 0.87]  ← speaker 1 silent, speakers 2 and 3 active
frame 1   → model → [0.91, 0.88, 0.04]  ← speakers 1 and 2 active, speaker 3 silent
frame 2   → model → [0.03, 0.92, 0.11]  ← only speaker 2 active
...
frame T   → model → [0.89, 0.07, 0.91]  ← speakers 1 and 3 active
```

Notice frame 0 has two speakers active simultaneously — EEND handles **overlapping speech**, which your current model can't do at all. Each of the 3 output values is an independent probability (hence sigmoid, not softmax) — multiple speakers can be active at the same time.

The final diarization output is a timeline:
```
0-1s:   Speaker 2 + Speaker 3 talking over each other
1-2s:   Speaker 1 + Speaker 2
2-3s:   Speaker 2 alone
...
```

That's what makes EEND useful for real conversations — people interrupt and talk over each other constantly.

---

So to directly answer your question: yes, you are predicting per time frame, not per clip. The `(batch, 3)` output you've been using is a simpler proxy task (counting speakers) that has been useful for learning. The real EEND task is `(batch, T, 3)` — who is speaking at every moment.

This also explains why self-attention is necessary: to decide if speaker 1 is active at frame 150, the model needs to compare frame 150's voice characteristics against frames throughout the recording to confirm it's the same person heard earlier.